# NLP Base 
# Content Based Recommendation

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Download dataset From Kaggle tmdb_5000

In [2]:
movies =pd.read_csv('tmdb_5000_credits.csv')
credits = pd.read_csv('tmdb_5000_movies.csv')

In [3]:
movies.shape

(4803, 4)

In [4]:
movies = movies.merge(credits,on='title')

In [5]:
movies.info()

<class 'pandas.DataFrame'>
RangeIndex: 4809 entries, 0 to 4808
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   movie_id              4809 non-null   int64  
 1   title                 4809 non-null   str    
 2   cast                  4809 non-null   str    
 3   crew                  4809 non-null   str    
 4   budget                4809 non-null   int64  
 5   genres                4809 non-null   str    
 6   homepage              1713 non-null   str    
 7   id                    4809 non-null   int64  
 8   keywords              4809 non-null   str    
 9   original_language     4809 non-null   str    
 10  original_title        4809 non-null   str    
 11  overview              4806 non-null   str    
 12  popularity            4809 non-null   float64
 13  production_companies  4809 non-null   str    
 14  production_countries  4809 non-null   str    
 15  release_date          4808 non-n

In [6]:
movies = movies[['movie_id','title','overview','genres','keywords','cast','crew']]

In [7]:
#movies.duplicated().sum()
movies.isna().sum()

movie_id    0
title       0
overview    3
genres      0
keywords    0
cast        0
crew        0
dtype: int64

In [8]:
movies.dropna(inplace=True)

In [9]:
movies.loc[0,'genres']

'[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]'

# Conver Genres into a list

In [10]:
import ast

def convert(obj):
    L = []
    for i in ast.literal_eval(obj):
        L.append(i['name'])
    return L



In [11]:
movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)


In [12]:

def convert2(obj):
    counter = 0
    L = []
    for i in ast.literal_eval(obj):
        if counter != 3:
            L.append(i['name'])
            counter+=1
        else:
            break
        
    return L



In [13]:
movies['cast'] = movies['cast'].apply(convert2)

In [14]:
def fetch_director(obj):
    L = []
    for i in ast.literal_eval(obj):
        if i['job'] == 'Director':
            L.append(i['name'])
    return L

In [15]:
movies['crew'] = movies['crew'].apply(fetch_director)

In [16]:
movies['overview'] = movies['overview'].apply(lambda x:x.split())

In [17]:
#movies.head(4)
movies['genres']= movies['genres'].apply(lambda x:[i.replace(" ","")for i in x])
movies['keywords']= movies['keywords'].apply(lambda x:[i.replace(" ","")for i in x])
movies['cast']= movies['cast'].apply(lambda x:[i.replace(" ","")for i in x])
movies['crew']= movies['crew'].apply(lambda x:[i.replace(" ","")for i in x])

In [18]:
#movies.head(3)
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

In [19]:
movies_df = movies[['movie_id','title','tags']]

In [20]:
movies_df['tags'] = movies_df['tags'].apply(lambda x:" ".join(x))

In [21]:
movies_df['tags'].apply(lambda x: x.lower())

0       in the 22nd century, a paraplegic marine is di...
1       captain barbossa, long believed to be dead, ha...
2       a cryptic message from bond’s past sends him o...
3       following the death of district attorney harve...
4       john carter is a war-weary, former military ca...
                              ...                        
4804    el mariachi just wants to play his guitar and ...
4805    a newlywed couple's honeymoon is upended by th...
4806    "signed, sealed, delivered" introduces a dedic...
4807    when ambitious new york attorney sam is sent t...
4808    ever since the second grade when he first saw ...
Name: tags, Length: 4806, dtype: str

# removing similar type duplicacy

In [38]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=5000,stop_words='english')
vectors =cv.fit_transform(movies_df['tags']).toarray()

In [39]:
import nltk

from nltk.stem.porter import PorterStemmer
ps =PorterStemmer()

In [28]:
def stem(text):
    y = []
    for i in text.split():
        y.append(ps.stem(i))
        
    return " ".join(y)

In [29]:
movies_df['tags'] = movies_df['tags'].apply(stem)

In [44]:
from sklearn.metrics.pairwise import cosine_similarity
similarity = cosine_similarity(vectors)
similarity

array([[1.        , 0.08346223, 0.0860309 , ..., 0.04499213, 0.        ,
        0.        ],
       [0.08346223, 1.        , 0.06063391, ..., 0.02378257, 0.        ,
        0.02615329],
       [0.0860309 , 0.06063391, 1.        , ..., 0.02451452, 0.        ,
        0.        ],
       ...,
       [0.04499213, 0.02378257, 0.02451452, ..., 1.        , 0.03962144,
        0.04229549],
       [0.        , 0.        , 0.        , ..., 0.03962144, 1.        ,
        0.08714204],
       [0.        , 0.02615329, 0.        , ..., 0.04229549, 0.08714204,
        1.        ]], shape=(4806, 4806))

In [45]:
def recommend(movie):
    movie_index = movies_df[movies_df['title'] == movie].index[0]
    distances = similarity[movie_index]
    movies_list = sorted(list(enumerate(distances)),reverse=True,key=lambda x:x[1])[1:6]
    
    for i in movies_list:
        print(movies_df.iloc[i[0]].title)

In [46]:
recommend("Avatar")

Aliens vs Predator: Requiem
Aliens
Falcon Rising
Independence Day
Titan A.E.
